# Automated Performance Outlook — PLN UP3 Cilacap

Notebook Colab untuk **development & demo cepat** via Ngrok tunnel.

> **Ingat:** URL Ngrok dari Colab **tidak permanen**. Untuk URL tetap, deploy `app.py` ke Streamlit Community Cloud dari repo GitHub. Notebook ini cocok utk iterasi cepat, demo ke atasan sebelum deploy final, atau kalibrasi koordinat.

**Prasyarat:**
1. Akun Ngrok gratis → ambil authtoken di https://dashboard.ngrok.com/get-started/your-authtoken
2. Repo project ini sudah lengkap (app.py, utils.py, config.py, assets/)

## Cell 1 — Install dependencies

Hanya perlu dijalankan sekali per sesi Colab.

In [ ]:
!pip install -q streamlit==1.39.0 pandas==2.2.3 Pillow==11.0.0 requests==2.32.3 pyngrok==7.2.0

## Cell 2 — Import libraries

In [ ]:
import os
import subprocess
import time
from pathlib import Path

from pyngrok import ngrok, conf as ngrok_conf

## Cell 3 — Clone repo project

Ganti URL kalau repo kamu private / pakai fork.

In [ ]:
REPO_URL = "https://github.com/surthe49-hub/dashboard-pln-cilacap.git"
PROJECT_DIR = "/content/dashboard-pln-cilacap"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo sudah ada, skip clone. Kalau mau pull terbaru:")
    print(f"  !cd {PROJECT_DIR} && git pull")

os.chdir(PROJECT_DIR)
!ls -la

## Cell 4 — Set Ngrok authtoken

Paste authtoken kamu di variabel `NGROK_TOKEN`. Jangan commit token ke Git.

In [ ]:
NGROK_TOKEN = "PASTE_TOKEN_DISINI"   # <-- ganti

if NGROK_TOKEN == "PASTE_TOKEN_DISINI":
    raise ValueError("Set NGROK_TOKEN dulu. Ambil di https://dashboard.ngrok.com")

ngrok_conf.get_default().auth_token = NGROK_TOKEN
print("Ngrok siap.")

## Cell 5 — Sanity check: data pipeline

Sebelum menyalakan server, verifikasi bahwa CSV bisa diambil dan renderer bekerja. Kalau cell ini error, perbaiki dulu sebelum lanjut.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from utils import fetch_sheet, get_row_for_month, extract_values, format_all, render_image

df = fetch_sheet()
print("Kolom terbaca:", list(df.columns))
print("\nSample 3 baris pertama:")
df.head(3)

## Cell 6 — Test render untuk 1 bulan

Kalau angka meleset dari ikonnya, jangan kaget — itu masalah kalibrasi koordinat, bukan bug. Kalibrasi di UI Streamlit (Developer Mode) di cell berikutnya.

In [ ]:
from IPython.display import display

TEST_MONTH = "Januari"   # ganti kalau mau bulan lain

row = get_row_for_month(df, TEST_MONTH)
if row is None:
    print(f"Bulan '{TEST_MONTH}' tidak ada di sheet.")
else:
    values = extract_values(row)
    formatted = format_all(values)
    print("Nilai terformat:")
    for k, v in formatted.items():
        print(f"  {k:25s} = {v}")
    img = render_image(formatted)
    display(img)

## Cell 7 — Kill proses lama (kalau ada)

Kalau kamu re-run notebook, ada kemungkinan Streamlit/Ngrok lama masih hidup. Bersihkan dulu.

In [ ]:
!pkill -f streamlit || true
ngrok.kill()
time.sleep(2)
print("Bersih.")

## Cell 8 — Jalankan Streamlit di background

Streamlit jalan di port 8501, background, dengan log ke `streamlit.log`.

In [ ]:
STREAMLIT_PORT = 8501

proc = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", str(STREAMLIT_PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

# Kasih Streamlit waktu utk bind port
time.sleep(5)
print(f"Streamlit PID: {proc.pid}")
print("Log (5 baris terakhir):")
!tail -5 streamlit.log

## Cell 9 — Buka tunnel Ngrok

URL yang keluar di cell ini adalah URL publik dashboard kamu. **Sementara** — mati saat sesi Colab berakhir.

In [ ]:
public_url = ngrok.connect(STREAMLIT_PORT, "http")
print("=" * 60)
print(f"  Dashboard live di: {public_url.public_url}")
print("=" * 60)

## Cell 10 — Monitor / stop

Jalankan cell ini kalau perlu:
- Cek log Streamlit (misalnya error di app)
- Matikan tunnel + Streamlit

In [ ]:
# Cek log terbaru
!tail -30 streamlit.log

In [ ]:
# Uncomment utk stop everything
# ngrok.disconnect(public_url.public_url)
# ngrok.kill()
# !pkill -f streamlit
# print("Semua service dimatikan.")